In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/mlx-session-zero/test_df_1.csv
/kaggle/input/competitions/mlx-session-zero/train_df_1.csv


In [2]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor
import lightgbm as lgb

# ── 1. LOAD ──────────────────────────────────────────────────
TRAIN_PATH = "/kaggle/input/competitions/mlx-session-zero/train_df_1.csv"
TEST_PATH  = "/kaggle/input/competitions/mlx-session-zero/test_df_1.csv"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print("Train:", train.shape, "| Test:", test.shape)

TARGET   = "Radiation"
test_ids = test["ID"].copy()
y        = train[TARGET].values.astype(float)

train_unix = train["UNIXTime"].values.astype(np.int64)
test_unix  = test["UNIXTime"].values.astype(np.int64)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))


# ── 2. TEMPORAL KNN FEATURES ─────────────────────────────────
# Radiation autocorrelation = 0.963 at 5-min intervals.
# Every test point has a train neighbour within ~5 minutes.
# This is the single biggest feature group.

def build_knn_features(query_unix, ref_unix, ref_rad, k=20, loo=False):
    n_feat = k + 8
    out    = np.zeros((len(query_unix), n_feat))
    for i in range(len(query_unix)):
        dists = np.abs(ref_unix - query_unix[i]).astype(float)
        if loo:
            dists[i] = 1e18
        nn_idx = np.argsort(dists)[:k]
        nn_rad = ref_rad[nn_idx]
        nd     = dists[nn_idx]
        w_exp  = np.exp(-nd / 300.0);  w_exp /= (w_exp.sum() + 1e-9)
        w_inv  = 1.0 / (nd + 1.0);    w_inv  /= (w_inv.sum() + 1e-9)
        diffs  = ref_unix[nn_idx] - query_unix[i]
        pm     = diffs < 0;  nm = diffs > 0
        out[i, :k]   = nn_rad
        out[i, k+0]  = np.dot(w_exp, nn_rad)
        out[i, k+1]  = np.dot(w_inv, nn_rad)
        out[i, k+2]  = nn_rad.mean()
        out[i, k+3]  = nn_rad.std()
        out[i, k+4]  = nn_rad.max()
        out[i, k+5]  = nd[0]
        out[i, k+6]  = nn_rad[pm].mean() if pm.sum() > 0 else nn_rad.mean()
        out[i, k+7]  = nn_rad[nm].mean() if nm.sum() > 0 else nn_rad.mean()
    return out

K = 20
print("Building KNN features (train LOO)...")
train_knn = build_knn_features(train_unix, train_unix, y, k=K, loo=True)
print("Building KNN features (test)...")
test_knn  = build_knn_features(test_unix,  train_unix, y, k=K, loo=False)
knn_cols  = ([f"nn_{i}" for i in range(K)] +
             ["nn_exp","nn_inv","nn_mean","nn_std","nn_max","nn_dist","nn_prev","nn_next"])


# ── 3. INTERPOLATION FEATURES ────────────────────────────────
# Build rich prev/next features using sorted train timeline.

order    = np.argsort(train_unix)
t_sorted = train_unix[order]
y_sorted = y[order]

def build_interp_features(query_unix, ref_t, ref_y, loo=False):
    out = np.zeros((len(query_unix), 14))
    for qi, qt in enumerate(query_unix):
        pos = np.searchsorted(ref_t, qt)
        # LOO: skip self
        if loo and pos < len(ref_t) and ref_t[pos] == qt:
            bef_t = ref_t[:pos];  bef_y = ref_y[:pos]
            aft_t = ref_t[pos+1:]; aft_y = ref_y[pos+1:]
        else:
            bef_t = ref_t[:pos];  bef_y = ref_y[:pos]
            aft_t = ref_t[pos:];  aft_y = ref_y[pos:]

        has_b = len(bef_t) > 0;  has_a = len(aft_t) > 0

        tb1 = bef_t[-1] if has_b else qt;  yb1 = bef_y[-1] if has_b else 0;  db1 = qt-tb1 if has_b else 1e9
        tb2 = bef_t[-2] if len(bef_t)>=2 else tb1;  yb2 = bef_y[-2] if len(bef_t)>=2 else yb1;  db2 = qt-tb2
        ta1 = aft_t[0]  if has_a else qt;  ya1 = aft_y[0]  if has_a else 0;  da1 = ta1-qt if has_a else 1e9
        ta2 = aft_t[1]  if len(aft_t)>=2 else ta1;  ya2 = aft_y[1]  if len(aft_t)>=2 else ya1;  da2 = ta2-qt

        # Linear interp
        if has_b and has_a:
            span = ta1 - tb1
            frac = (qt - tb1) / (span + 1e-9)
            lin1 = yb1 + frac * (ya1 - yb1)
        else:
            span = 1e9;  frac = 0.5;  lin1 = yb1 if has_b else ya1

        # Weighted averages
        w1 = np.exp(-db1/300.0); w2 = np.exp(-db2/300.0); w3 = np.exp(-da1/300.0)
        wavg3 = (w1*yb1 + w2*yb2 + w3*ya1) / (w1+w2+w3+1e-9)
        wi1=1/(db1+1); wi2=1/(db2+1); wi3=1/(da1+1); wi4=1/(da2+1)
        wavg4 = (wi1*yb1+wi2*yb2+wi3*ya1+wi4*ya2) / (wi1+wi2+wi3+wi4)

        # Quadratic interp (3-point: prev2, prev1, next1)
        if len(bef_t) >= 2 and has_a:
            x0,x1,x2 = tb2,tb1,ta1;  y0,y1,y2 = yb2,yb1,ya1
            denom = (x0-x1)*(x0-x2)*(x1-x2) + 1e-9
            a_ = (x2*(y1-y0)+x1*(y0-y2)+x0*(y2-y1)) / denom
            b_ = (x2**2*(y0-y1)+x1**2*(y2-y0)+x0**2*(y1-y2)) / denom
            c_ = (x1*x2*(x1-x2)*y0+x2*x0*(x2-x0)*y1+x0*x1*(x0-x1)*y2) / denom
            quad = a_*qt**2 + b_*qt + c_
        else:
            quad = lin1

        out[qi, 0]  = yb1;    out[qi, 1]  = ya1
        out[qi, 2]  = yb2;    out[qi, 3]  = ya2
        out[qi, 4]  = lin1;   out[qi, 5]  = wavg3
        out[qi, 6]  = wavg4;  out[qi, 7]  = quad
        out[qi, 8]  = span;   out[qi, 9]  = frac
        out[qi, 10] = db1;    out[qi, 11] = da1
        out[qi, 12] = (yb1+ya1)/2
        out[qi, 13] = ya1 - yb1  # trend direction
    return out

print("Building interpolation features (train LOO)...")
train_interp = build_interp_features(train_unix, t_sorted, y_sorted, loo=True)
print("Building interpolation features (test)...")
test_interp  = build_interp_features(test_unix,  t_sorted, y_sorted, loo=False)
interp_cols  = ["prev_rad","next_rad","prev2_rad","next2_rad",
                "lin_interp","wavg3","wavg4","quad_interp",
                "span","frac","dist_prev","dist_next","midpoint","trend"]

print("LOO RMSE of interpolation methods:")
for i, n in enumerate(interp_cols[:8]):
    print(f"  {n}: {rmse(y, np.clip(train_interp[:,i],0,None)):.4f}")


# ── 4. FEATURE ENGINEERING ───────────────────────────────────

def parse_time(t):
    try: h,m,s = str(t).strip().split(":"); return int(h)*3600+int(m)*60+int(s)
    except: return np.nan

def engineer(df):
    df = df.copy()
    df["obs_sec"]      = df["Time"].apply(parse_time)
    df["hour"]         = df["obs_sec"] // 3600
    df["minute"]       = (df["obs_sec"] % 3600) // 60
    df["sunrise_sec"]  = df["TimeSunRise"].apply(parse_time)
    df["sunset_sec"]   = df["TimeSunSet"].apply(parse_time)
    df["daylight"]     = df["sunset_sec"] - df["sunrise_sec"]
    df["solar_noon"]   = (df["sunrise_sec"] + df["sunset_sec"]) / 2
    df["since_sunrise"]= df["obs_sec"] - df["sunrise_sec"]
    df["until_sunset"] = df["sunset_sec"] - df["obs_sec"]
    df["is_daytime"]   = ((df["obs_sec"]>=df["sunrise_sec"])&(df["obs_sec"]<=df["sunset_sec"])).astype(int)
    df["daylight_frac"]= (df["since_sunrise"]/(df["daylight"]+1e-9)).clip(0,1)
    df["solar_sin"]    = np.sin(np.pi*df["daylight_frac"])*df["is_daytime"]
    df["solar_sin_sq"] = df["solar_sin"]**2
    df["solar_sin_cu"] = df["solar_sin"]**3
    df["mins_from_noon"]= np.abs(df["obs_sec"]-df["solar_noon"])/60
    df["hour_sin"]     = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]     = np.cos(2*np.pi*df["hour"]/24)
    df["min_of_day"]   = df["hour"]*60 + df["minute"]
    df["mod_sin"]      = np.sin(2*np.pi*df["min_of_day"]/1440)
    df["mod_cos"]      = np.cos(2*np.pi*df["min_of_day"]/1440)
    df["unix_in_day"]  = df["UNIXTime"] % 86400
    df["Data_dt"]      = pd.to_datetime(df["Data"], format="%d-%m-%Y", errors="coerce")
    df["month"]        = df["Data_dt"].dt.month
    df["day"]          = df["Data_dt"].dt.day
    df["dayofyear"]    = df["Data_dt"].dt.dayofyear
    df["month_sin"]    = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"]    = np.cos(2*np.pi*df["month"]/12)
    df["doy_sin"]      = np.sin(2*np.pi*df["dayofyear"]/365)
    df["doy_cos"]      = np.cos(2*np.pi*df["dayofyear"]/365)
    df["sunrise_hr"]   = df["sunrise_sec"]/3600
    df["sunset_hr"]    = df["sunset_sec"]/3600
    df["daylight_hr"]  = df["daylight"]/3600
    df["wind_sin"]     = np.sin(np.deg2rad(df["WindDirection(Degrees)"]))
    df["wind_cos"]     = np.cos(np.deg2rad(df["WindDirection(Degrees)"]))
    df["wind_u"]       = -df["Speed"]*df["wind_sin"]
    df["wind_v"]       = -df["Speed"]*df["wind_cos"]
    df["temp_sq"]      = df["Temperature"]**2
    df["hum_sq"]       = df["Humidity"]**2
    df["clarity"]      = (100-df["Humidity"])*df["Pressure"]/100
    df["dewpoint"]     = df["Temperature"] - ((100-df["Humidity"])/5)
    df["temp_dew"]     = df["Temperature"] - df["dewpoint"]
    df["temp_over_hum"]= df["Temperature"]/(df["Humidity"]+1)
    df["solar_x_temp"] = df["solar_sin"]*df["Temperature"]
    df["solar_x_hum"]  = df["solar_sin"]*(100-df["Humidity"])
    df["solar_x_clear"]= df["solar_sin"]*df["clarity"]
    df["elev_x_temp"]  = df["solar_sin_sq"]*df["Temperature"]
    df["elev_x_clear"] = df["solar_sin_sq"]*df["clarity"]
    df["sin3_x_temp"]  = df["solar_sin_cu"]*df["Temperature"]
    return df

train_fe = engineer(train)
test_fe  = engineer(test)

BASE = ["Temperature","Pressure","Humidity","Speed","WindDirection(Degrees)",
        "temp_sq","hum_sq","clarity","dewpoint","temp_dew","temp_over_hum",
        "is_daytime","solar_sin","solar_sin_sq","solar_sin_cu",
        "daylight_frac","daylight","daylight_hr","since_sunrise",
        "until_sunset","solar_noon","mins_from_noon",
        "hour","minute","min_of_day","obs_sec","hour_sin","hour_cos",
        "mod_sin","mod_cos","unix_in_day",
        "month","day","dayofyear","month_sin","month_cos","doy_sin","doy_cos",
        "sunrise_sec","sunset_sec","sunrise_hr","sunset_hr",
        "wind_sin","wind_cos","wind_u","wind_v",
        "solar_x_temp","solar_x_hum","solar_x_clear",
        "elev_x_temp","elev_x_clear","sin3_x_temp"]
BASE = [c for c in BASE if c in train_fe.columns]

med       = train_fe[BASE].median()
X_b       = train_fe[BASE].fillna(med).reset_index(drop=True)
X_b_test  = test_fe[BASE].fillna(med).reset_index(drop=True)

X = pd.concat([X_b,
               pd.DataFrame(train_knn,   columns=knn_cols),
               pd.DataFrame(train_interp, columns=interp_cols)], axis=1)
Xt= pd.concat([X_b_test,
               pd.DataFrame(test_knn,   columns=knn_cols),
               pd.DataFrame(test_interp, columns=interp_cols)], axis=1)

print(f"\nTotal features: {X.shape[1]}")


# ── 5. LIGHTGBM ───────────────────────────────────────────────
print("\n" + "="*60)
print("Training LightGBM...")

lgb_params = {
    "objective":"regression_l2","metric":"rmse",
    "num_leaves":512,"learning_rate":0.02,
    "feature_fraction":0.70,"bagging_fraction":0.70,"bagging_freq":5,
    "min_child_samples":10,"reg_alpha":0.05,"reg_lambda":0.05,
    "n_estimators":4000,"random_state":42,"verbose":-1,"n_jobs":-1,
}

N = 5
kf = KFold(n_splits=N, shuffle=True, random_state=42)
oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(Xt))

for fold, (tr, val) in enumerate(kf.split(X, y), 1):
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[val], y[val])],
          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(5000)])
    oof_lgb[val]  = m.predict(X.iloc[val])
    test_lgb     += m.predict(Xt) / N
    print(f"  Fold {fold}: {rmse(y[val], oof_lgb[val]):.5f}  iters={m.best_iteration_}")

lgb_score = rmse(y, oof_lgb)
print(f"  LightGBM OOF RMSE: {lgb_score:.5f}")


# ── 6. EXTRATREES (3 seeds) ───────────────────────────────────
print("\n" + "="*60)
print("Training ExtraTrees (3 seeds)...")

oof_et_all  = []
test_et_all = []

for seed in [11, 22, 33]:
    oof_s  = np.zeros(len(X))
    test_s = np.zeros(len(Xt))
    for fold, (tr, val) in enumerate(kf.split(X, y), 1):
        m = ExtraTreesRegressor(n_estimators=600, max_features=0.80,
                                min_samples_leaf=1, bootstrap=False,
                                random_state=seed, n_jobs=-1)
        m.fit(X.iloc[tr], y[tr])
        oof_s[val]  = m.predict(X.iloc[val])
        test_s     += m.predict(Xt) / N
    et_s = rmse(y, oof_s)
    print(f"  ET seed={seed}  OOF: {et_s:.5f}")
    oof_et_all.append(oof_s)
    test_et_all.append(test_s)

oof_et  = np.mean(oof_et_all,  axis=0)
test_et = np.mean(test_et_all, axis=0)
et_score = rmse(y, oof_et)
print(f"  ExtraTrees avg OOF: {et_score:.5f}")


# ── 7. ADAPTIVE BLEND ────────────────────────────────────────
# Key: for points with short interpolation span, trust interp more.
# For points with long span, trust ML model more.
print("\n" + "="*60)
print("Finding optimal blend...")

# Static blend: LGB + ET
best_s, best_w = 1e9, 0.5
for w in np.arange(0, 1.01, 0.02):
    r = rmse(y, np.clip(w*oof_lgb + (1-w)*oof_et, 0, None))
    if r < best_s: best_s, best_w = r, w
print(f"  Best static LGB+ET: LGB={best_w:.2f}  RMSE={best_s:.5f}")

# Span-adaptive: blend interpolation with model
lin_oof  = np.clip(train_interp[:,4], 0, None)  # lin_interp
lin_test = np.clip(test_interp[:,4],  0, None)
spans_tr = train_interp[:,8]  # span column
spans_te = test_interp[:,8]

ml_oof  = best_w * oof_lgb  + (1 - best_w) * oof_et
ml_test = best_w * test_lgb + (1 - best_w) * test_et

best_a, best_tau = 1e9, 500
for tau in [200, 300, 500, 800, 1000, 1500, 2000]:
    w_i = np.exp(-spans_tr / tau)
    blend = np.clip(w_i * lin_oof + (1 - w_i) * ml_oof, 0, None)
    r = rmse(y, blend)
    if r < best_a: best_a, best_tau = r, tau
print(f"  Best adaptive tau={best_tau}  RMSE={best_a:.5f}")

# Final prediction
w_i_te   = np.exp(-spans_te / best_tau)
final    = np.clip(w_i_te * lin_test + (1 - w_i_te) * ml_test, 0, None)

print(f"\n{'='*60}")
print(f"FINAL SCORES:")
print(f"  LightGBM  OOF: {lgb_score:.5f}")
print(f"  ExtraTrees OOF: {et_score:.5f}")
print(f"  Static blend:   {best_s:.5f}")
print(f"  Adaptive blend: {best_a:.5f}")
print(f"  Your current LB: 68.311  |  #1: 68.248")
print(f"{'='*60}")


# ── 8. SAVE ALL SUBMISSIONS ──────────────────────────────────

def save_sub(fname, pred):
    pred = np.clip(pred, 0, None)
    pd.DataFrame({"ID": test_ids, "TARGET": pred}).to_csv(fname, index=False)
    print(f"  Saved {fname}  mean={pred.mean():.2f}")

print("\nSaving submissions...")

# 1. MAIN — adaptive blend (submit this first)
save_sub("submission.csv", final)

# 2. LightGBM only
save_sub("sub_lgb.csv", test_lgb)

# 3. ExtraTrees only
save_sub("sub_et.csv", test_et)

# 4. Static ML blend
save_sub("sub_ml_blend.csv", ml_test)

# 5. Heavy interp blends
save_sub("sub_80interp_20ml.csv", np.clip(0.80*lin_test + 0.20*ml_test, 0, None))
save_sub("sub_70interp_30ml.csv", np.clip(0.70*lin_test + 0.30*ml_test, 0, None))
save_sub("sub_60interp_40ml.csv", np.clip(0.60*lin_test + 0.40*ml_test, 0, None))

# 6. wavg4 blend (4-neighbor weighted average + model)
wavg4_test = np.clip(test_interp[:,6], 0, None)
save_sub("sub_wavg4_ml.csv", np.clip(0.3*wavg4_test + 0.7*ml_test, 0, None))

print("\nAll files:")
for f in sorted(os.listdir("/kaggle/working")):
    if f.endswith(".csv"):
        print(f"  {f}")

Train: (20004, 12) | Test: (3334, 11)
Building KNN features (train LOO)...
Building KNN features (test)...
Building interpolation features (train LOO)...
Building interpolation features (test)...
LOO RMSE of interpolation methods:
  prev_rad: 94.9813
  next_rad: 93.6487
  prev2_rad: 109.2056
  next2_rad: 108.9591
  lin_interp: 75.0710
  wavg3: 75.4582
  wavg4: 71.9945
  quad_interp: 407638331986030.9375

Total features: 94

Training LightGBM...
  Fold 1: 72.95300  iters=218
  Fold 2: 70.40764  iters=193
  Fold 3: 68.11163  iters=195
  Fold 4: 70.72122  iters=241
  Fold 5: 74.52670  iters=211
  LightGBM OOF RMSE: 71.37811

Training ExtraTrees (3 seeds)...
  ET seed=11  OOF: 69.87543
  ET seed=22  OOF: 69.83250
  ET seed=33  OOF: 69.92293
  ExtraTrees avg OOF: 69.83584

Finding optimal blend...
  Best static LGB+ET: LGB=0.00  RMSE=69.83732
  Best adaptive tau=500  RMSE=69.66247

FINAL SCORES:
  LightGBM  OOF: 71.37811
  ExtraTrees OOF: 69.83584
  Static blend:   69.83732
  Adaptive blend